# Sync to Snowflake

Creates/refreshes the `ICEBERG TABLE` objects for one layer's tables, per the
`layer` widget (`raw`/`curated`/`analytics`), as listed under that key in
`snowflake_sync_config.json`.

## How it runs

- Runs as **three separate job tasks** — one per layer — right after that layer's
  own pipeline, not once at the end of the job.
- Against the catalog integration `create_snowflake_catalog_integration.ipynb`
  creates once; this notebook **never creates or modifies the integration itself**
  — only run that one manually, on its own, when (re)bootstrapping.

## Change detection

Before refreshing an already-existing Iceberg table, checks whether its Databricks
source actually changed in the layer's pipeline's latest update, and skips the
`ALTER ICEBERG TABLE ... REFRESH` call if not. A table being created for the first
time is always refreshed once regardless.

- **One query per layer, not per table** — a single `event_log(pipeline_id)` call
  (`GROUP BY flow_name`) covers every table in the layer at once.
- **`pipeline_id` comes from the job, not runtime discovery** — the DAB bundle
  passes it in directly via `resources.pipelines.*.id`; no job/task introspection
  needed.
- **Streaming tables and materialized views need different signals** (verified
  against the real event log, not assumed):
  - A streaming table's row-count metrics are incremental — a positive sum means
    real new data.
  - A materialized view's `num_output_rows` is always the *full* recomputed row
    count, so it's useless as a change signal there. Materialized views are
    checked via the `planning_information` event's chosen maintenance technique
    instead (`MAINTENANCE_TYPE_NO_OP` means nothing changed).

In [ ]:
import snowflake.connector

dbutils.widgets.text("catalog_name", "f1_analytics")
dbutils.widgets.text("secret_scope", "snowflake-iceberg-sync")
dbutils.widgets.text("sync_config_file", "snowflake_sync_config.json")
dbutils.widgets.text("layer", "raw")
dbutils.widgets.text("pipeline_id", "")

catalog_name = dbutils.widgets.get("catalog_name")
secret_scope = dbutils.widgets.get("secret_scope")
sync_config_file = dbutils.widgets.get("sync_config_file")
layer = dbutils.widgets.get("layer")
pipeline_id = dbutils.widgets.get("pipeline_id")

# Must match the formula create_snowflake_catalog_integration.ipynb used to create
# this integration — derived from catalog_name alone, nothing passed between notebooks.
catalog_integration_name = f"uc_{catalog_name}_iceberg"

print(f"Catalog: {catalog_name}")
print(f"Catalog integration: {catalog_integration_name}")
print(f"Sync config file: {sync_config_file}")
print(f"Layer: {layer}")
print(f"Pipeline id: {pipeline_id or '(none — change detection disabled, every table will be refreshed)'}")

In [ ]:
# All credentials come from the Databricks secret scope Terraform populates.
# Nothing sensitive is ever hard-coded here or passed as a plain job parameter.
snowflake_account = dbutils.secrets.get(secret_scope, "snowflake-account")
snowflake_user = dbutils.secrets.get(secret_scope, "snowflake-service-user")
snowflake_private_key_pem = dbutils.secrets.get(secret_scope, "snowflake-service-user-private-key")
snowflake_role = dbutils.secrets.get(secret_scope, "snowflake-role")
snowflake_warehouse = dbutils.secrets.get(secret_scope, "snowflake-warehouse")
snowflake_database = dbutils.secrets.get(secret_scope, "snowflake-database")

In [ ]:
from cryptography.hazmat.primitives import serialization

# snowflake-connector-python wants the key as DER bytes, not the PEM string stored in the secret scope.
private_key_der = serialization.load_pem_private_key(
    snowflake_private_key_pem.encode(), password=None
).private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)

conn = snowflake.connector.connect(
    account=snowflake_account,
    user=snowflake_user,
    private_key=private_key_der,
    role=snowflake_role,
    warehouse=snowflake_warehouse,
    database=snowflake_database,
)
cur = conn.cursor()

In [ ]:
import json
import os

# The notebook's own workspace path tells us where sync_config_file (deployed
# alongside this notebook by the bundle, via the repo's normal file sync — no
# separate upload step) lives, resolved at runtime rather than hard-coded, since
# dev/prod deploy to different workspace paths.
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = os.path.dirname(notebook_path)
if not notebook_dir.startswith("/Workspace"):
    notebook_dir = f"/Workspace{notebook_dir}"
config_path = f"{notebook_dir}/{sync_config_file}"

with open(config_path) as f:
    sync_tables = json.load(f)["layers"][layer]

print(f"Loaded {len(sync_tables)} table mapping(s) for layer '{layer}' from {config_path}")

In [ ]:
# One event_log(pipeline_id) call, GROUP BY flow_name, covers every table in this
# layer's pipeline at once — not one event_log() call per table. Verified against
# this project's real event log (run as the owning SP, which event_log() requires):
#   * origin.flow_name is fully-qualified (catalog.schema.table), sometimes
#     backtick-quoted; dataset_name is NULL, so flow_name is what's matched on.
#   * Each update emits QUEUED/PLANNING/STARTING/RUNNING/COMPLETED events PLUS
#     status-less metrics-only flow_progress events — the final status must ignore
#     those NULLs rather than just taking the single latest-timestamp row.
#   * Streaming tables: a COMPLETED flow that wrote nothing reports metrics = {}.
#     When it writes, num_output_rows (plus upserted/deleted for CDC flows) appears
#     and is incremental.
#   * Materialized views: num_output_rows is the FULL recomputed row count, so it's
#     always positive and useless as a change signal. The planning_information
#     event's chosen maintenance technique is used instead: NO_OP means nothing to
#     refresh.
event_data = {}  # keyed by lowercase flow_name (= lowercase catalog.schema.table)

if pipeline_id:
    rows = spark.sql(f"""
        WITH latest_uid AS (
            SELECT max_by(origin.update_id, timestamp) AS uid
            FROM event_log("{pipeline_id}")
            WHERE event_type = 'flow_progress'
        )
        SELECT
            lower(regexp_replace(origin.flow_name, '`', ''))   AS flow_name,
            origin.update_id                                    AS update_id,
            max_by(details:flow_progress.status, timestamp)
                FILTER (WHERE event_type = 'flow_progress'
                          AND details:flow_progress.status IS NOT NULL)  AS final_status,
            sum(coalesce(CAST(details:flow_progress.metrics.num_output_rows   AS BIGINT), 0)
              + coalesce(CAST(details:flow_progress.metrics.num_upserted_rows AS BIGINT), 0)
              + coalesce(CAST(details:flow_progress.metrics.num_deleted_rows  AS BIGINT), 0))
                FILTER (WHERE event_type = 'flow_progress')                 AS rows_changed,
            count(*) FILTER (WHERE event_type = 'planning_information') > 0 AS is_mv,
            max_by(details, timestamp)
                FILTER (WHERE event_type = 'planning_information')          AS planning_raw
        FROM event_log("{pipeline_id}")
        WHERE origin.update_id = (SELECT uid FROM latest_uid)
          AND event_type IN ('flow_progress', 'planning_information')
        GROUP BY flow_name, origin.update_id
    """).collect()

    for row in rows:
        r = row.asDict()
        flow = r["flow_name"]

        # MV only: technique_information is an array of candidate techniques; the
        # one the planner actually used has is_chosen = true.
        planning_raw = r.pop("planning_raw")
        r["mv_maintenance_type"] = None
        if planning_raw:
            techniques = json.loads(planning_raw).get("planning_information", {}).get("technique_information", [])
            r["mv_maintenance_type"] = next(
                (t.get("maintenance_type") for t in techniques if t.get("is_chosen")), None)

        if r["final_status"] != "COMPLETED":
            r["source_changed"], r["reason"] = True, f"final status {r['final_status']} -> assume changed"
        elif r["is_mv"]:
            mt = (r["mv_maintenance_type"] or "").upper()
            r["source_changed"] = "NO_OP" not in mt
            r["reason"] = f"MV technique={mt or 'UNKNOWN'}" + (" -> no change" if not r["source_changed"] else " -> changed")
        else:
            r["source_changed"] = (r["rows_changed"] or 0) > 0
            r["reason"] = f"streaming table rows written/upserted/deleted = {r['rows_changed'] or 0}"

        event_data[flow] = r

print(f"Loaded change-detection data for {len(event_data)} flow(s) from pipeline {pipeline_id or '(none)'}")
print(json.dumps(event_data, indent=2, default=str))

In [ ]:
# AUTO_REFRESH and manual REFRESH are mutually exclusive in Snowflake (enabling one
# blocks the other) — each layer's sync task runs on-demand right after that layer's
# own pipeline finishes, so every table is created WITHOUT auto-refresh and
# explicitly REFRESHed below, but only when the source actually had something new.
#
# SHOW ICEBERG TABLES IN SCHEMA scopes each existence check to exactly the target
# schema(s) this layer uses, explicitly qualified as database.schema — no reliance
# on the session's current database/schema (there isn't one; the connection above
# doesn't pin a schema), and no need to scan every schema in the database.
target_schemas = sorted({entry["target_schema"] for entry in sync_tables})
existing_tables = set()
for schema in target_schemas:
    cur.execute(f'SHOW ICEBERG TABLES IN SCHEMA "{snowflake_database}"."{schema}"')
    result_columns = [c[0] for c in cur.description]
    schema_col = result_columns.index("schema_name")
    name_col = result_columns.index("name")
    existing_tables.update(
        (row[schema_col].lower(), row[name_col].lower()) for row in cur.fetchall()
    )

# The Unity Catalog source schema for every entry in this layer's list is the layer
# itself (snowflake_sync_config.json groups tables by Unity Catalog schema name).
source_schema = layer

def create_iceberg_table(table_fqn, source_table, source_schema):
    cur.execute(f"""
        CREATE ICEBERG TABLE {table_fqn}
        CATALOG = '{catalog_integration_name}'
        CATALOG_TABLE_NAME = '{source_table}'
        CATALOG_NAMESPACE = '{source_schema}'
    """)

for entry in sync_tables:
    source_table = entry["source_table"]
    target_schema = entry["target_schema"]
    target_table = entry["target_table"]
    table_fqn = f'"{snowflake_database}"."{target_schema}"."{target_table}"'
    source_fqn = f"{catalog_name}.{source_schema}.{source_table}"

    # Lowercased before comparing — Snowflake's SHOW output and this config's casing
    # don't need to match exactly, only case-insensitively.
    table_exists = (target_schema.lower(), target_table.lower()) in existing_tables

    change = event_data.get(source_fqn.lower()) or {
        "source_changed": True,
        "reason": "no event data found for this table -> assume changed",
    }
    print(f"{source_fqn}: {change['reason']}")

    if not table_exists:
        print(f"Creating Iceberg table {table_fqn}")
        create_iceberg_table(table_fqn, source_table, source_schema)

    # A newly-created table is always refreshed once; an existing one only
    # refreshes if its Databricks source changed in the layer's pipeline's latest
    # update.
    if not table_exists or change["source_changed"]:
        print(f"Refreshing Iceberg table {table_fqn}")
        try:
            cur.execute(f"ALTER ICEBERG TABLE {table_fqn} REFRESH")
        except snowflake.connector.errors.ProgrammingError as e:
            # A full refresh of the Databricks source (dp pipeline full_refresh,
            # or a checkpoint reset) recreates its underlying Delta/Iceberg table,
            # which changes that table's Iceberg UUID. Snowflake pinned the OLD
            # UUID when this ICEBERG TABLE object was first created, and a plain
            # REFRESH can't reconcile a changed UUID -- only dropping and
            # recreating the object picks up the new one. Narrowed to this exact
            # error so any other REFRESH failure still surfaces normally instead
            # of being silently papered over.
            if "does not match the table UUID" in str(e):
                print(f"UUID mismatch for {table_fqn} (source was full-refreshed) — recreating instead of refreshing")
                cur.execute(f"DROP ICEBERG TABLE {table_fqn}")
                create_iceberg_table(table_fqn, source_table, source_schema)
            else:
                raise
    else:
        print(f"Skipping refresh for {table_fqn} — source unchanged since last update")

In [ ]:
cur.close()
conn.close()